# 01 Generate SOCAT ID Intermediates

Purpose: generate the SOCAT observational intrinsic-dimension CSV intermediates used by the plotting scripts.

Inputs:
- pCO2 LEAP reconstruction zarr
- SOCAT mask zarr

Outputs:
- `SOCATobs_dim_all.csv`
- `SOCATobs_dim_input.csv`
- `SOCATobs_scales_all.csv`
- `SOCATobs_scales_input.csv`
- `SOCATobs_full_dim_all.csv`
- `SOCATobs_full_scales_all.csv`

Estimated runtime: slow, because DADAPy computes distance matrices for the sampled and full SOCAT datasets.

Notes:
- This notebook is provenance material. This optional regeneration step requires the raw data and a compatible Python environment.
- The figures use the saved files in `intermediates/`.


In [ ]:
from pathlib import Path

import pandas as pd
import xarray as xr
from dadapy import Data
from sklearn.preprocessing import StandardScaler

# Local paths; replace with gs:// paths if running against cloud storage.
RECONSTRUCTION_ZARR = Path("../raw/pCO2_LEAP_fco2-residual-full-dataset-preML_198201-202412.zarr")
SOCAT_MASK_ZARR = Path("../raw/socat_mask_feb1982-dec2022.zarr")
OUTPUT_DIR = Path("../intermediates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLE = 16000
RANDOM_STATE = 10
TIME_SLICE = slice("2020-01-01", "2022-12-31")
MAXK_CAP = 10000
RANGE_MAX = 1024


## Feature Definitions

The feature and target definitions match the SOCAT intrinsic-dimension workflow.


In [ ]:
FEATURES = [
    "sst",
    "sst_anomaly",
    "sss",
    "sss_anomaly",
    "chl_log",
    "chl_log_anomaly",
    "mld_log",
    "xco2_trend",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"


## Helper Functions


In [ ]:
def load_socat_dataframe(reconstruction_zarr, socat_mask_zarr):
    socat_mask = xr.open_dataset(socat_mask_zarr, engine="zarr")
    ds = xr.open_dataset(reconstruction_zarr, engine="zarr")
    ds = ds.sel(time=slice("1982-02-01", "2022-12-31"))

    aligned_mask = socat_mask.reindex(time=ds.time, method="nearest")
    ds = ds.where(aligned_mask.socat_mask == 1)
    ds[TARGET] = ds["fco2"] - ds["xco2_trend"]
    ds = ds[FEATURES + [TARGET]]
    ds = ds.sel(time=TIME_SLICE)

    return ds.to_dataframe().dropna()

def scale_socat(frame):
    scaled = StandardScaler().fit_transform(frame.loc[:, FEATURES + [TARGET]])
    return pd.DataFrame(scaled, columns=FEATURES + [TARGET])

def compute_id_tables(scaled_frame, include_input_only=True):
    data_all = Data(scaled_frame.to_numpy(), verbose=False)
    data_all.compute_distances(maxk=min(scaled_frame.shape[0] - 1, MAXK_CAP))
    ids_all, _, scales_all = data_all.return_id_scaling_gride(range_max=RANGE_MAX)

    if not include_input_only:
        return ids_all, None, scales_all, None

    input_frame = scaled_frame.drop([TARGET], axis=1)
    data_input = Data(input_frame.to_numpy(), verbose=False)
    data_input.compute_distances(maxk=min(input_frame.shape[0] - 1, MAXK_CAP))
    ids_input, _, scales_input = data_input.return_id_scaling_gride(range_max=RANGE_MAX)

    return ids_all, ids_input, scales_all, scales_input


## Load SOCAT Observations

This selects SOCAT-observed points from 2020-2022 and builds the residual target `delta_fco2_1D`.


In [ ]:
SOCAT = load_socat_dataframe(RECONSTRUCTION_ZARR, SOCAT_MASK_ZARR)
SOCAT.shape


## Generate 16k SOCAT ID Tables

Outputs are used by:
- `scripts/plot_id_socat_models.py`
- `scripts/plot_socat_id_and_summary.py`
- `scripts/plot_comparisons.py`


In [ ]:
SOCAT_sample = SOCAT.sample(n=N_SAMPLE, random_state=RANDOM_STATE)
SOCAT_scaled_sample = scale_socat(SOCAT_sample)

ids_all, ids_input, scales_all, scales_input = compute_id_tables(
    SOCAT_scaled_sample,
    include_input_only=True,
)

pd.DataFrame({"SOCAT": ids_all}).to_csv(OUTPUT_DIR / "SOCATobs_dim_all.csv", index=False)
pd.DataFrame({"SOCAT": ids_input}).to_csv(OUTPUT_DIR / "SOCATobs_dim_input.csv", index=False)
pd.DataFrame({"SOCAT": scales_all}).to_csv(OUTPUT_DIR / "SOCATobs_scales_all.csv", index=False)
pd.DataFrame({"SOCAT": scales_input}).to_csv(OUTPUT_DIR / "SOCATobs_scales_input.csv", index=False)


## Generate Full SOCAT ID Tables

This uses all available 2020-2022 SOCAT rows after masking and dropping missing values.


In [ ]:
SOCAT_scaled_full = scale_socat(SOCAT)
ids_full, _, scales_full, _ = compute_id_tables(
    SOCAT_scaled_full,
    include_input_only=False,
)

pd.DataFrame({"SOCAT": ids_full}).to_csv(OUTPUT_DIR / "SOCATobs_full_dim_all.csv", index=False)
pd.DataFrame({"SOCAT": scales_full}).to_csv(OUTPUT_DIR / "SOCATobs_full_scales_all.csv", index=False)


## Outputs Created

After running this notebook, these files should exist in `intermediates/`:

- `SOCATobs_dim_all.csv`
- `SOCATobs_dim_input.csv`
- `SOCATobs_scales_all.csv`
- `SOCATobs_scales_input.csv`
- `SOCATobs_full_dim_all.csv`
- `SOCATobs_full_scales_all.csv`

Quick check:


In [ ]:
expected_outputs = [
    "SOCATobs_dim_all.csv",
    "SOCATobs_dim_input.csv",
    "SOCATobs_scales_all.csv",
    "SOCATobs_scales_input.csv",
    "SOCATobs_full_dim_all.csv",
    "SOCATobs_full_scales_all.csv",
]

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(filename, "OK" if path.exists() else "MISSING")
